# Kom i gang med otdiag

Kjører hele kjeden på simulerte data: tcpdump-linjer → statistikk per minutt → avvik → diagnose.
Kjør `python tools/gen_testdata.py` først (fra rotmappen).

In [ ]:
import sys, csv
sys.path.insert(0, '..')
import pandas as pd
from datetime import datetime
from otdiag import parse_tcpdump_line, aggregate_minutes, build_baseline, detect, diagnose, HostEvent

In [ ]:
with open('../testdata/coresim_tcpdump.log') as f:
    stats = list(aggregate_minutes(p for p in map(parse_tcpdump_line, f) if p))
df = pd.DataFrame(s.as_row() for s in stats)
df['_time'] = pd.to_datetime(df['_time'])
df.head()

In [ ]:
# Pakker per minutt per kilde
df.pivot_table(index='_time', columns='src_ip', values='packets', aggfunc='sum').plot(logy=True, figsize=(12, 5), legend=False);

In [ ]:
# Baseline fra de første 40 minuttene (normaldrift)
start = stats[0].minute
normal = [s for s in stats if (s.minute - start).total_seconds() < 40 * 60]
baseline = build_baseline(normal)
pd.DataFrame(baseline.to_rows()).head(15)

In [ ]:
rest = [s for s in stats if (s.minute - start).total_seconds() >= 40 * 60]
anomalies = detect(rest, baseline, k=4)
pd.DataFrame(a.as_row() for a in anomalies).groupby('rule').size()

In [ ]:
hosts = [HostEvent(time=datetime.fromisoformat(r['time']), host_ip=r['host_ip'], kind=r['kind'], detail=r['detail'],
                   user=r['user'] or None, suspicious=r['suspicious'] == 'True')
         for r in csv.DictReader(open('../testdata/host_events.csv'))]
incidents = diagnose(anomalies, hosts)
pd.DataFrame(i.as_row() for i in incidents)[['_time', 'severity', 'diagnose', 'target', 'explanation']]

In [ ]:
# Sammenlign med fasit
pd.read_csv('../testdata/attack_windows.csv')

## Ideer videre
- Prøv k = 3, 4, 5, 6 og tell oppdagede angrep vs. falske alarmer (resultatdel i rapporten).
- Tren en klassifikator (scikit-learn) på avvikene for å gjette angrepstype, og sammenlign med regel-diagnosen.